## Production_Coding -> Logging -> type hint -> dotenv 

##### **I am going to practice using logging instead of print statements because that's what we use in production. I'll also practice type hints and python-dotenv.**

#### Logging:

In [ ]:
# logging
import logging
logging.basicConfig(level=logging.INFO,format='%(asctime)s - %(levelname)s - %(messages)s') # %(name)s, %(filename)s,%(lineno)s
logger = logging.getLogger(__name__)
logger.debug("It will run in dev time")
logger.info("App started successfully")
logger.warning("Config file is missing,using default")
logger.error("API call failed",exc_info=True) # exc_info show the error with full history of the app function
logger.critical("Database down")

### LEVELS(severity Order):DEBUG < INFO < WARNING < ERROR < CRITICAL

**Handlers**

In [ ]:
import logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

In [ ]:
# in real-time log file exceeds more than 10mb it will create new log file auromatically
from logging.handlers import RotatingFileHandler

handler = RotatingFileHandler("app.log", maxBytes=10_000_000, backupCount=5)

#### Type-Hints:

In [ ]:
from typing import List, Dict, Optional, Union
def get_scores(names:List[str]) -> Dict[str,int]:
    return {name: len(name) for name in names}
def find_user(user_id:int) -> Optional[str]:
    #optional either it will return str or none 
    if user_id == 1:
        return "aswath"
    return None
def process(value: Union[str,int]) -> str:
    # Union returns int or str
    return str(value)

#### Dot-env:

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
print(api_key)

##### Type Annotations - Recap + Deep dive

In [ ]:
#Annotation = broader concept of typing - It fix the "type label" for variable/function 
name: str = "Aswath"          # variable annotation
age: int = 25
scores: list[int] = [90, 85]

def greet(name: str, age: int = 25) -> str:   # function annotation
    return f"{name} is {age}"

#### dataclass - "Auto-generate boring code"

In [ ]:
# if we use dataclass decorator we dont want to mention the __init__,__repr__,__eq__ functions because it automatically generate that

from dataclasses import dataclass
@dataclass
class User:
    name: str
    age: int
u = User("Aswath",25)
print(u)
print(u.name)

In [ ]:
from dataclasses import dataclass,field
@dataclass
class Config:
    name: str
    log_level:str ="INFO"
    tags:list[str]=field(default_factory=list) # list and dict are mutable object so we cannot give more default values so we are using default_factory

#### Pydantic - "Dataclass + Validation"

##### why we are using pydantic means in data class it wont validate the input it only type-hint it but pydantic do the both things

In [ ]:
from pydantic import BaseModel,Field # Field-> extra validation rule

class User(BaseModel):
    name:str
    age:int
    api_request:int = Field(default=1000,gt=0)
u=User(name="aswath",age=25,api_request=100)
print(u)

In [ ]:
# Real_time EDA scenerio:
from pydantic import BaseModel,Field
from typing import Optional

class api_config(BaseModel):
    api_key:str
    model_name:str="claude-haiku"
    max_token:int = Field(default=10000,gt=0)
    temperature:Optional[float]=None

app = api_config(api_key="abcdef",max_token=500)
print(app.model_dump()) # convert object to dict format

** Internal, trusted data → dataclass. External/untrusted data (user input, API response, config files) → pydantic.**

##### Task 1:


In [ ]:
from dataclasses import dataclass
from typing import Optional
import logging

logging.basicConfig(level=logging.INFO,format='%(asctime)s-%(levelname)s-%(name)s-%(message)s')
logger = logging.getLogger(__name__)

@dataclass
class Student:
    name: str
    marks: int
    id: Optional[int] = None
    def login(self):
        logger.info("Successfully login:{self.name}")

student1 = Student(name="aswath",marks=85)
logger.info(f"Student created:{student1}")

student1.login()


##### Task-2

In [ ]:
import logging
from pydantic import BaseModel, ValidationError

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s-%(levelname)s-%(name)s-%(message)s'
)
logger = logging.getLogger(__name__)

class Employee(BaseModel):
    name: str
    salary: float
    department: str


# ---- Valid data test ----
try:
    emp1 = Employee(name="Aswath", salary=50000.0, department="Engineering")
    logger.info(f"Employee created successfully: {emp1}")
except ValidationError:
    logger.error("Validation failed for emp1", exc_info=True)


# ---- Invalid data test (salary ku string kuduthu) ----
try:
    emp2 = Employee(name="Ravi", salary="fifty thousand", department="HR")
    logger.info(f"Employee created successfully: {emp2}")
except ValidationError:
    logger.error("Validation failed for emp2", exc_info=True)

##### Task-3:

In [ ]:
# ---- .env file ----
# API_KEY=abc123xyz
# MODEL_NAME=claude-haiku
# MAX_TOKENS=1000

import os
import logging
from typing import Optional
from pydantic import BaseModel, Field, ValidationError
from dotenv import load_dotenv

load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s-%(levelname)s-%(name)s-%(message)s'
)
logger = logging.getLogger(__name__)


class AppConfig(BaseModel):
    api_key: str
    model_name: str = "claude-haiku"          # default fallback
    max_tokens: int = Field(default=1000, gt=0)


# .env la irundhu values edukurom
env_api_key: Optional[str] = os.getenv("API_KEY")
env_model_name: Optional[str] = os.getenv("MODEL_NAME")
env_max_tokens: Optional[str] = os.getenv("MAX_TOKENS")

try:
    # env_api_key missing na, Pydantic automatic ah error throw pannum
    # (api_key ku default illa, so compulsory field)
    config = AppConfig(
        api_key=env_api_key,
        model_name=env_model_name if env_model_name else "claude-haiku",
        max_tokens=int(env_max_tokens) if env_max_tokens else 1000
    )
    logger.info(f"Config loaded successfully: {config}")

except ValidationError:
    logger.critical("Failed to load AppConfig from .env", exc_info=True)